In [0]:
import os
import math
import pickle
import numpy as np
import torch
from contextlib import nullcontext

from model import GPTConfig
from model_baseline import BaselineGPT
from model_rope import GPTWithRoPE

from evaluate_test_supervised import get_batch

In [0]:
config_file = "config/enwik8_char_rope_trm.py"
# config_file = "config/enwik8_char_rope_baselineV2.py"
config = {}
with open(config_file, 'r') as f:
    exec(f.read(), {}, config)

config = {k: v for k, v in config.items() if not k.startswith('__')}
assert 0 < config['gradient_accumulation_steps'] <= 1, "gradient_accumulation_steps > 1 is not supported in supervised training."
model_path_temp = "/dbfs/tmp/enwik8"
config["out_dir"] += "_nohlt9V3_2trns4rec_sclnolrnrm2.0"
config["out_dir"] = os.path.join(model_path_temp, config['out_dir'])
checkpoint_path = os.path.join(config['out_dir'], 'ckpt.pt')

In [0]:
device = config['device']
device_type = 'cuda' if 'cuda' in device else 'cpu'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[config['dtype']]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

data_dir = os.path.join('data', config['dataset'])
meta_path = os.path.join(data_dir, 'meta.pkl')
with open(meta_path, 'rb') as f:
    meta = pickle.load(f)
vocab_size = meta['vocab_size']
config['vocab_size'] = vocab_size

gpt_config_keys = ['n_layer', 'n_head', 'n_embd', 'block_size', 'bias', 'vocab_size', 'dropout']
gpt_config = {k: v for k, v in config.items() if k in gpt_config_keys}
gptconf = GPTConfig(**gpt_config)

if config.get('model_type') in ('rope', 'nope'):
    model = GPTWithRoPE(gptconf)
    print("Using GPTWithRoPE model.")
elif config.get('model_type') == 'trm':
    from model_rope_trm import TRMGPTWithRoPE
    model = TRMGPTWithRoPE(gptconf)
    print("Using TRMGPTWithRoPE model.")
else:
    model = BaselineGPT(gptconf)
    print("Using BaselineGPT model.")
model.to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model'], strict=False)
model.eval()

test_data_path = os.path.join(data_dir, 'test.bin')
test_data = np.memmap(test_data_path, dtype=np.uint16, mode='r')
test_data = torch.from_numpy(test_data.astype(np.int64))

block_size = config['block_size']
batch_size = config.get('batch_size', 64)
N_supervised_steps = config.get('N_supervised_steps_eval', 2)

num_tokens = len(test_data) - 1
x_tokens = test_data[:num_tokens]
y_tokens = test_data[1:num_tokens+1]
num_batches = num_tokens // block_size
x_tokens = x_tokens[:num_batches * block_size]
y_tokens = y_tokens[:num_batches * block_size]
x_batches = x_tokens.view(-1, block_size)
y_batches = y_tokens.view(-1, block_size)

val_dataset = torch.utils.data.TensorDataset(x_batches, y_batches)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

@torch.inference_mode()
def estimate_test_loss():
    losses = np.zeros(N_supervised_steps)
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        for X, Y in val_loader:
            X = X.to(device)
            Y = Y.to(device)
            z_H, z_L = None, None
            for step in range(N_supervised_steps):
                with ctx:
                    logits, loss, z_H, z_L, _ = model(X, Y, z_H, z_L)
                losses[step] += loss.item()
    mean_loss = losses / len(val_loader)
    bpc = mean_loss / math.log(2)
    for step in range(N_supervised_steps):
        print(f"Steps {step}: {mean_loss[step]:.4f} | bpc: {bpc[step]:8.3f}")

estimate_test_loss()

In [0]:
print(model.a_L.item() , model.a_H.item(), model.a_X.item(), model.b_L.item(), model.b_H.item())

In [0]:
# N_supe (3, 3): Test loss: 3.2052 | test_bpc:    4.624
# N_supe (3, 2): Test loss: 3.1893 | test_bpc:    4.601

In [0]:
# N_supe (16, 3): Test loss: 3.1903 | test_bpc:    4.603
# N_supe (16, 4): Test loss: 3.2028 | test_bpc:    4.621
# N_supe (16, 6): Test loss: 3.2070 | test_bpc:    4.627
# N_supe (16, 10): Test loss: 3.2159 | test_bpc:    4.640
# N_supe (16, 15): Test loss: 3.2547 | test_bpc:    4.696
# N_supe (16, 16): Test loss: 3.2650 | test_bpc:    4.710

In [0]:
# N_supe (5, 4): Test loss: 2.3786 | test_bpc:    3.432
# N_supe (5, 5): Test loss: 2.3904 | test_bpc:    3.449
# N_supe (5, 16): Test loss: 2.8633 | test_bpc:    4.131
# Halt
# N_supe (5, 4): Test loss: 2.3961 | test_bpc:    3.457
# N_supe (5, 5): Test loss: 2.4096 | test_bpc:    3.476

In [0]:
# Halt
# N_supe (6, 3): Test loss: 2.3102 | test_bpc:    3.333
# N_supe (6, 4): Test loss: 2.2987 | test_bpc:    3.316
# N_supe (6, 5): Test loss: 2.3035 | test_bpc:    3.323
# N_supe (6, 6): Test loss: 2.3202 | test_bpc:    3.347
# Halt 256emb
# N_supe (6, 4): Test loss: 2.5947 | test_bpc:    3.743
# N_supe (6, 5): Test loss: 2.5947 | test_bpc:    3.743
# N_supe (6, 6): Test loss: 2.6002 | test_bpc:    3.751
# Halt 256emb, 1.5*steps
# N_supe (6, 4): Test loss: 2.5707 | test_bpc:    3.709
# N_supe (6, 5): Test loss: 2.5710 | test_bpc:    3.709
# N_supe (6, 6): Test loss: 2.5784 | test_bpc:    3.720

# Halt 256emb, 1.5*steps
# N_supe (8, 5): Test loss: 2.4948 | test_bpc:    3.599
# N_supe (8, 6): Test loss: 2.4942 | test_bpc:    3.598
# N_supe (8, 7): Test loss: 2.4985 | test_bpc:    3.605

# Halt 256emb, 1.5*steps
# N_supe (10, 5): Test loss: 2.3768 | test_bpc:    3.429
# N_supe (10, 6): Test loss: 2.3729 | test_bpc:    3.423
# N_supe (10, 7): Test loss: 2.3731 | test_bpc:    3.424
# N_supe (10, 8): Test loss: 2.3764 | test_bpc:    3.428
# N_supe (10, 9): Test loss: 2.3820 | test_bpc:    3.437

# Halt2.0 256emb, 1.5*steps  
# N_supe (10, 7): Test loss: 2.3738 | test_bpc:    3.425    
# N_supe (10, 8): Test loss: 2.3760 | test_bpc:    3.428   

# Halt2.0 (lmb=1.0) 256emb, 1.5*steps  
# N_supe (10, 7): Test loss: 2.4178 | test_bpc:    3.488
# N_supe (10, 8): Test loss: 2.4188 | test_bpc:    3.490

# Halt2.0 256emb, 3.0*steps  
# N_supe (10, 6): Test loss: 2.3842 | test_bpc:    3.440
# N_supe (10, 7): Test loss: 2.3830 | test_bpc:    3.438    
# N_supe (10, 8): Test loss: 2.3856 | test_bpc:    3.442    
# N_supe (10, 9): Test loss: 2.3913 | test_bpc:    3.450  

# N_supe (10, 8): Test loss: 2.4189 | test_bpc:    3.490
# N_supe (10, 9): Test loss: 2.4312 | test_bpc:    3.507

# Halt2.0 256emb, 1.5*steps, 512cntxt_v2
# N_supe (6, 4): Test loss: 2.6938 | test_bpc:    3.886 
# N_supe (6, 5): Test loss: 2.6943 | test_bpc:    3.887  

# Halt2.0 256emb, 1.5*steps, 256cntxt
# N_supe (6, 5): Test loss: 2.6337 | test_bpc:    3.800

# Halt2.0 256emb, 1.5*steps, 256cntxt_4trns
# N_supe (6, 4): Test loss: 2.6758 | test_bpc:    3.860
# N_supe (6, 5): Test loss: 2.6714 | test_bpc:    3.854
# N_supe (6, 6): Test loss: 2.6733 | test_bpc:    3.857

# Halt2.0 256emb, 1.5*steps, 256cntxt_4trns_4rec
# N_supe (6, 3): Test loss: 2.6161 | test_bpc:    3.774
# N_supe (6, 4): Test loss: 2.6074 | test_bpc:    3.762
# N_supe (6, 5): Test loss: 2.6072 | test_bpc:    3.761
# N_supe (6, 6): Test loss: 2.6072 | test_bpc:    3.769

# 1.0Halt2.0 256emb, 1.5*steps, 256cntxt_4trns_4rec
# N_supe (6, 4): Test loss: 2.7440 | test_bpc:    3.959

# noHalt2.0 256emb, 1.5*steps, 256cntxt_4trns_4rec
# N_supe (6, 4): Test loss: 2.6072 | test_bpc:    3.761
# N_supe (6, 5): Test loss: 2.6044 | test_bpc:    3.757

# noHalt2.0 512emb, 1.5*steps, 256cntxt_4trns_4rec
# N_supe (6, 3): Test loss: 2.2743 | test_bpc:    3.281
# N_supe (6, 4): Test loss: 2.2589 | test_bpc:    3.259
# N_supe (6, 5): Test loss: 2.2597 | test_bpc:    3.260

# noHalt2.0 1024emb, 1.5*steps, 256cntxt_4trns_4rec
# N_supe (6, 3): Test loss: 2.0164 | test_bpc:    2.909
# N_supe (6, 4): Test loss: 2.0052 | test_bpc:    2.893
# N_supe (6, 5): Test loss: 2.0144 | test_bpc:    2.906

# noHalt2.0 1024emb, 1.5*steps, 256cntxt_2trns_4rec
# N_supe (6, 4): Test loss: 2.0295 | test_bpc:    2.928
# N_supe (6, 5): Test loss: 2.0375 | test_bpc:    2.940

# noHalt2.0 1024emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmtkn
# N_supe (6, 4): Test loss: 2.0441 | test_bpc:    2.949
# N_supe (6, 5): Test loss: 2.0550 | test_bpc:    2.965

# noHalt2.0 1024emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmnodrptkn
# N_supe (6, 4): Test loss: 2.0237 | test_bpc:    2.920
# N_supe (6, 5): Test loss: 2.0373 | test_bpc:    2.939

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmnodrptkn
# N_supe (6, 4): Test loss: 2.3546 | test_bpc:    3.397
# N_supe (6, 5): Test loss: 2.3642 | test_bpc:    3.411

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmnodrptkn
# N_supe (6, 3): Test loss: 2.1815 | test_bpc:    3.147
# N_supe (6, 4): Test loss: 2.1772 | test_bpc:    3.141
# N_supe (6, 5): Test loss: 2.1871 | test_bpc:    3.155

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmnodrptkn
# N_supe (6, 3): Test loss: 2.1738 | test_bpc:    3.136
# N_supe (6, 4): Test loss: 2.1735 | test_bpc:    3.136
# N_supe (6, 5): Test loss: 2.1801 | test_bpc:    3.145

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmnodrptkn
# N_supe (6, 4): Test loss: 2.3711 | test_bpc:    3.421


# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmdrptkn2.0
# N_supe (6, 2): Test loss: 2.1569 | test_bpc:    3.112
# N_supe (6, 3): Test loss: 2.1386 | test_bpc:    3.085
# N_supe (6, 4): Test loss: 2.1426 | test_bpc:    3.091

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmdrptkn3.0
# Steps (6, 0): 2.0321 | bpc:    2.932
# Steps (6, 1): 1.9352 | bpc:    2.792
# Steps (6, 2): 1.9389 | bpc:    2.797
# Steps (6, 3): 1.9404 | bpc:    2.799

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmdrptkn4.0
# Steps (6, 0): 1.9624 | bpc:    2.831
# Steps (6, 1): 1.9383 | bpc:    2.796
# Steps (6, 2): 1.9418 | bpc:    2.801
# Steps (6, 3): 1.9437 | bpc:    2.804

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmdrptkn3.0
Steps 8, 0: 2.3388 | bpc:    3.374
Steps 1: 1.9162 | bpc:    2.765
Steps 2: 1.9137 | bpc:    2.761
Steps 3: 1.9170 | bpc:    2.766
Steps 4: 1.9192 | bpc:    2.769
Steps 5: 1.9204 | bpc:    2.771

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmdrptkn3.0_nolnrm
Steps 8, 0: 2.0879 | bpc:    3.012
Steps 1: 2.0216 | bpc:    2.917
Steps 2: 2.0268 | bpc:    2.924
Steps 3: 2.0321 | bpc:    2.932
Steps 4: 2.0357 | bpc:    2.937
Steps 5: 2.0381 | bpc:    2.940

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmdrptkn3.0_rmsnrm
Steps 8, 0: 2.2425 | bpc:    3.235
Steps 1: 1.9342 | bpc:    2.790
Steps 2: 1.9339 | bpc:    2.790
Steps 3: 1.9370 | bpc:    2.794
Steps 4: 1.9388 | bpc:    2.797
Steps 5: 1.9398 | bpc:    2.799

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_6rec_initfrmdrptkn3.0
Steps 0: 2.0488 | bpc:    2.956
Steps 1: 1.9245 | bpc:    2.777
Steps 2: 1.9264 | bpc:    2.779
Steps 3: 1.9294 | bpc:    2.784

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_initfrmdrptkn3.0
Steps 9, 0: 2.4242 | bpc:    3.497
Steps 1: 1.9150 | bpc:    2.763
Steps 2: 1.9122 | bpc:    2.759
Steps 3: 1.9140 | bpc:    2.761
Steps 4: 1.9153 | bpc:    2.763
Steps 5: 1.9159 | bpc:    2.764

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_2rec_initfrmdrptkn3.0
Steps 18, 0: 2.3598 | bpc:    3.404
Steps 1: 1.9664 | bpc:    2.837
Steps 2: 1.9427 | bpc:    2.803
Steps 3: 1.9310 | bpc:    2.786
Steps 4: 1.9244 | bpc:    2.776
Steps 5: 1.9210 | bpc:    2.771
Steps 6: 1.9194 | bpc:    2.769
Steps 7: 1.9187 | bpc:    2.768
Steps 8: 1.9184 | bpc:    2.768
Steps 9: 1.9183 | bpc:    2.767
Steps 10: 1.9182 | bpc:    2.767
Steps 11: 1.9182 | bpc:    2.767

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_2rec_initfrmdrptkn3.0
Steps 9, 0: 2.2415 | bpc:    3.234
Steps 1: 1.9486 | bpc:    2.811
Steps 2: 1.9400 | bpc:    2.799
Steps 3: 1.9380 | bpc:    2.796
Steps 4: 1.9378 | bpc:    2.796

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_scl
Steps 9, 0: 2.4242 | bpc:    3.497
Steps 1: 1.9150 | bpc:    2.763
Steps 2: 1.9122 | bpc:    2.759

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_sclnolrnrm
Steps 0: 2.9835 | bpc:    4.304
Steps 1: 1.9427 | bpc:    2.803
Steps 2: 1.9422 | bpc:    2.802

# noHalt2.0 512emb, 1.5*steps, 256cntxt_2trns_4rec_sclnolrnrm2.0
Steps 0: 2.0836 | bpc:    3.006
Steps 1: 1.9926 | bpc:    2.875